# Model Architecture

In [ ]:
import os
import sys

project_path = os.path.abspath("..")
sys.path.append(project_path)

from package.model import SEDNetSegmentLevel, MosbeatnetV2


import torch
import torchaudio
import torch.nn as nn
from torchinfo import summary


input_wav_test = "/home/mosbnet/mosdet/mosbeatnet_main/data/simulated_data/gen_exp1_indoor_species_sex/indoor_miru/forest/test/indoor_miru_forest_test_000001.wav"


waveform, sample_rate = torchaudio.load(input_wav_test)

print("Sample rate:", sample_rate)
print("Input shape:", waveform.shape)   # [channels, samples]
print("Input dtype:", waveform.dtype)
print("Input tensor:", waveform)


print("First 20 samples:", waveform)


# Data Preprocessing
segment_length = int(sample_rate * 0.5)  # 50 ms
num_samples = waveform.shape[-1]
remainder = num_samples % segment_length
print("Segment length:", segment_length)
print("Number of samples:", num_samples)
print("Remainder:", remainder)


segments = waveform.unfold(1, segment_length, segment_length)  # [channels, num_segments, segment_length]
print("Segments shape before permute:", segments.shape)  # [channels, num_segments, segment_length]
segments = segments.permute(1, 0, 2).contiguous()
print("Segments shape:", segments.shape)  # [num_segments, channels, segment_length]

model_input = segments.unsqueeze(0)
# segments:    [num_segments, 1, 4000]
# model_input: [1, num_segments, 1, 4000]
print("Model input shape:", model_input.shape)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
segment_sec = 0.5
clip_sec = 10.0

n_timesteps = int(clip_sec / segment_sec)  # 20

model = MosbeatnetV2(n_timesteps=n_timesteps,n_outputs=5).to(device)


model_input = model_input.to(device)

summary(
    model,
    input_data=model_input,
    col_names=(
        "input_size",
        "output_size",
        "num_params",
        "trainable",
    ),
    depth=4,
)




Sample rate: 8000
Input shape: torch.Size([1, 80000])
Input dtype: torch.float32
Input tensor: tensor([[ 0.0200,  0.0050, -0.0112,  ...,  0.0005, -0.0026, -0.0114]])
First 20 samples: tensor([[ 0.0200,  0.0050, -0.0112,  ...,  0.0005, -0.0026, -0.0114]])
Segment length: 4000
Number of samples: 80000
Remainder: 0
Segments shape before permute: torch.Size([1, 20, 4000])
Segments shape: torch.Size([20, 1, 4000])
Model input shape: torch.Size([1, 20, 1, 4000])


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
MosbeatnetV2                             [1, 20, 1, 4000]          [1, 20, 5]                --                        True
├─Sequential: 1-1                        [20, 1, 4000]             [20, 128, 111]            --                        True
│    └─ConvBlock1D: 2-1                  [20, 1, 4000]             [20, 32, 2000]            --                        True
│    │    └─Sequential: 3-1              [20, 1, 4000]             [20, 32, 2000]            --                        True
│    │    │    └─Conv1d: 4-1             [20, 1, 4000]             [20, 32, 2000]            480                       True
│    │    │    └─BatchNorm1d: 4-2        [20, 32, 2000]            [20, 32, 2000]            64                        True
│    │    │    └─GELU: 4-3               [20, 32, 2000]            [20, 32, 2000]            --                        --
│    